### Agentic RAG SYSTEM

In [55]:
import warnings 
warnings.filterwarnings('ignore')

# Document load 
from langchain_community.document_loaders import PyPDFLoader 
loader  = PyPDFLoader('Static GK 2025.pdf')
pages = loader.load()

In [56]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
import hashlib

# Split Data 

spliter = RecursiveCharacterTextSplitter(chunk_size=1400 , chunk_overlap=180)
text_spliter = spliter.split_documents(pages)
chunks = [i.page_content for i in text_spliter]
metadata = [i.metadata for i in text_spliter]
ids = [hashlib.md5(chunk.encode('utf-8')).hexdigest() for chunk in chunks]
print(f'print first 5 ids : {ids[:2]}')

print first 5 ids : ['df52eef7bfa55759b4642211e13e3020', '622d6c3b19974d6f39f9950848df1607']


In [57]:
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding_function = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# client and collection create 
client = chromadb.PersistentClient(path="./Agentic_RAG_Database")
collection = client.get_or_create_collection(name="Agentic_Rag",embedding_function=embedding_function)

if chunks:
    collection.add(
        ids=ids,
        documents=chunks , metadatas=metadata
    )
collection.count()

225

In [58]:
from langchain_ollama import ChatOllama 
llm = ChatOllama(model="qwen2.5:1.5b")

In [59]:
# Hybrid Corpus 
from rank_bm25 import BM25Okapi 
def tokenization(token):
    token = token.lower()
    token = token.split()
    return token 

tokens = [tokenization(i) for i in chunks]
bm_corpus = BM25Okapi(tokens)

print(f'sucussfully : {bm_corpus}')

sucussfully : <rank_bm25.BM25Okapi object at 0x12b945b50>


In [66]:
import ast
from langchain_community.tools import tool

@tool
def calculator(expression: str) -> str:
    """
    Evaluates arithmetic expressions.
    
    Args:
        expression: A mathematical expression string to compute, e.g. '45 + 150' or '25 * (4 + 2)'.
    """
    try:
        # Compile expression into an AST tree safely
        code = compile(expression, "<string>", "eval")
        
        # Reject any variable names, imports, or function calls
        for name in code.co_names:
            raise NameError(f"Use of variable/function '{name}' is not allowed")
            
        # Safely evaluate math without access to Python builtins
        result = eval(code, {"__builtins__": None}, {})
        return str(result)
    except Exception as e:
        return f"Calculation error: {str(e)}"
    
# retrival 
@tool
def Hybrid_Retrive(query:str):
    """THIS IS local given document by user . so , user asking all sort of questions is passing through this tool"""
    query_re = llm.invoke(f"write the query based on symentic search : {query}").content.strip()

    # Thats Vector DB retrival 
    result = collection.query(query_texts=[query_re] , n_results=5)
    dis  = result['distances'][0] 
    docs = result['documents'][0]
    threshold = 0.9
    print(f'the distance is : {dis}')
    dense_docs = []
    for i , d in zip(dis,docs):
        if threshold > i :
            dense_docs.append(d)
    # Thats Hybrid RAG Retrival using indexing 
    query_tokens = tokenization(query_re)
    score = bm_corpus.get_scores(query=query_tokens)
    def get_top_tokens (score , k=10):
        index = list(enumerate(score))
        idx_sorted = sorted(index,key=lambda x:x[1],reverse=True)
        return [doc for doc , _ in idx_sorted[:10]]
    index_tokens = [chunks[i] for i in get_top_tokens(score=score,k=10)]
    
    rrf_token = {}
    
    for rank , doc in enumerate(dense_docs):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
    for rank , doc in enumerate(index_tokens):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
        
    marge = sorted(rrf_token.items() , key = lambda x:x[1] , reverse=True)
    get_docs = [i for i , _ in marge[:5]]
    
    return "\n\n".join(get_docs)

In [67]:
tools = [calculator,Hybrid_Retrive]
tool_name  = {t.name : t for t in tools}
print(tool_name)

{'calculator': StructuredTool(name='calculator', description="Evaluates arithmetic expressions.\n\nArgs:\n    expression: A mathematical expression string to compute, e.g. '45 + 150' or '25 * (4 + 2)'.", args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x12b9c9260>), 'Hybrid_Retrive': StructuredTool(name='Hybrid_Retrive', description='THIS IS local given document by user . so , user asking all sort of questions is passing through this tool', args_schema=<class 'langchain_core.utils.pydantic.Hybrid_Retrive'>, func=<function Hybrid_Retrive at 0x12b6f09a0>)}


In [68]:
# Tool conection with LLM 
try:
    llm_tool_blind = llm.bind_tools(tools=tools)
    print(bool(llm_tool_blind))
except Exception as e :
    print(str(e))

True


In [69]:
def search_tool(question: str):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]
    
    # 1. Initialize context list at the top (matching name used later)
    retrieved_contexts = []
    
    response = llm_tool_blind.invoke(messages)
    
    # Direct answer path (No tool calls)
    if not response.tool_calls:
        ans = response.content if response.content else "no response generate"
        return ans, ["N/A - direct answer"]
    
    messages.append(response)
    
    # Tool execution path
    for call in response.tool_calls:
        name = call['name']
        arguments = call['args']
        
        tool_response = tool_name[name].invoke(arguments)
        
        # Capture context chunks from Hybrid_Retrive
        if name == "Hybrid_Retrive":
            chunks = [c.strip() for c in str(tool_response).split("\n\n") if c.strip()]
            retrieved_contexts.extend(chunks)
        
        messages.append({
            "tool_call_id": call['id'],
            "content": str(tool_response),
            "role": "tool"
        })
    
    # 2. Fallback check for non-retrieval tools (e.g., calculator)
    if not retrieved_contexts:
        retrieved_contexts = ["N/A - Math Calculation"]
        
    result = llm.invoke(messages)
    ans = result.content if result.content else "no response generate"
    
    return ans, retrieved_contexts

In [70]:
from datasets import Dataset 
from ragas import evaluate 
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from langchain_community.embeddings import HuggingFaceEmbeddings


test_cases = [
    {
        "question": "who is first First Chief of Army Staff",
        "ground_truth": "General Maharaj Rajendra Singh Ji was the first Chief of Army Staff."
    },
    {
        "question": "Calculate 45 * 12 + 150",
        "ground_truth": "690"
    }
]

user_input = []
retrive_context = []
response =  []
reference = [] 

for item in test_cases:
    qus = item['question']
    truth = item['ground_truth']
    
    answer , context = search_tool(question=qus)
    print(answer)
    
    user_input.append(qus)
    retrive_context.append(context)
    response.append(answer)
    reference.append(truth)
    
data = {
    "user_input":user_input , 
    "retrieved_contexts":retrive_context ,
    "response":response , 
    "reference":reference
}

embedding = HuggingFaceEmbeddings(model_name= "all-MiniLM-L6-v2")
dataset = Dataset.from_dict(data)

result = evaluate(
    dataset = dataset , 
    metrics=[Faithfulness(), AnswerRelevancy(), ContextPrecision(), ContextRecall()] , 
    embeddings=embedding , 
    llm=llm
)

print(result)

I apologize, but I'm not able to retrieve information on who the first Chief of Army Staff was as that would typically require accessing historical records and documents. I recommend consulting a historical or military authority for accurate information.
The calculation is correct. The result of 45 * 12 + 150 is 690.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

{'faithfulness': 0.0000, 'answer_relevancy': 0.4376, 'context_precision': 0.0000, 'context_recall': 1.0000}
